<a href="https://colab.research.google.com/github/Tokoloshe714/26COA122CW/blob/toolRent/menu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [96]:
"""
Most of the following i got from AI as i did not know how to remove the sample data folder that appears automatically every new runtime. I need to remove the file as I am not permitted more than 10 files for submition.
Note this is not directly intended to be part of my coursework, but is needed."""

def remove_colab_sample_data():
    import shutil
    shutil.rmtree("/content/sample_data/", ignore_errors=True)

In [97]:
"""
toolSearch and associated functions

Last edited on: 2026/11/09
            By: F613550
Cell description:
    This cell contains four(4) functions.
    • search_setup reads all the tools from the file Tool_Info.txt line by
      line. Each tool is stored as an element of a list that has the tool attributes in a dictionary.
    • find_query searches the tool list for matches against a chosen attribute
      using fuzzy string matching and returns the best results.
    • if_new_query is the callback that runs whenever the search box or
      dropdown changes. It clears the previous results and displays the new matches.
    • toolSearch builds the interactive search UI and connects the widgets so
      the search updates live.
"""


def tool_info_setup():

    global tool_info_list

    # Define a list[dict{}]. Better than many r/w to .txt file
    tool_info_list = []

    try:
        with open("Tool_Info.txt", "r") as file:
            for line in file:

                line = line.strip()

                if not line: # accounts for empty lines
                    continue

                # copies each attribute into a list
                line_info = [att.strip() for att in line.split(",")]

                if len(line_info) == 5:
                    tool_info_list.append({
                        "ID": line_info[0],
                        "Name": line_info[1],
                        "Size": line_info[2],
                        "Purchase Price": line_info[3],
                        "Purchase Date": line_info[4],
                    })

    except error as e:
        print(f"There was an error: {e}")

    return tool_info_list


def find_query(search_string: str,attribute: str, limit: int= 5):

    if not search_string.strip():
        return tool_info_list[:limit]
    else:

        option = [str(tool[attribute]) for tool in tool_info_list]

        # search for all items in tool_info_list according to options
        sorted_matches = process.extract(
            search_string,
            option,
            scorer=fuzz.WRatio, # method for string comparison
            limit=limit
        )
        # reformatting
        results = []
        for match_test, score, index in sorted_matches:
            tool = tool_info_list[index].copy()
            #tool["Score"] = score
            results.append(tool)
        # return a sorted list of all valid tools in the correct TYPE!!!
        return results



def if_new_query(change, tools_found):

    with tools_found:

        clear_output(wait=True)
        query = search_box.value
        attribute = attribute_dropdown.value
        results = find_query(query, attribute)

        if not results:
            print("No matches found.")
        else:
            import pandas as pd
            display(pd.DataFrame(results))


def toolSearch():

    global search_box, attribute_dropdown

    tool_attributes = ["ID", "Name", "Size", "Purchase Price",  "Purchase Date"]
    search_box = widgets.Text(
        value="",
        placeholder="Type to search...",
        description="Search:",
        continuous_update=True, # updates live while typing
        layout=widgets.Layout(width="400px")
    )

    attribute_dropdown = widgets.Dropdown(
        options=tool_attributes,
        value="Name",
        description="Search in:",
        layout=widgets.Layout(width="250px")
    )

    tools_found = widgets.Output()
    # globals ar'nt working. changed approach to use lambda's
    search_box.observe(lambda change: if_new_query(change, tools_found),    names="value")
    attribute_dropdown.observe(lambda change: if_new_query(change, tools_found), names="value")
    search_interface = widgets.VBox([
        widgets.HTML("Tool Search"),
        widgets.HBox([search_box, attribute_dropdown]), tools_found
        ])

    display(search_interface)
    if_new_query(None, tools_found= tools_found)


In [98]:
"""
subscriptions ---> will be moved to toolRent

Last edited on: 2026/11/11
            By: F613550
Cell description:
    Customers should be identified using their unique ID numbers. For simplicity, we use 4-digit integers (e.g.,
    1000-9999) for these IDs. Only customers with active    subscriptions, as verified by the provided
    ‘subscriptionManager.pyc’ module, are considered valid for renting tools. Your program should distinguish
    between valid and invalid IDs
"""

# CustomerID,SubscriptionType,StartDate,EndDate
# true for premium members


def subscription_setup():

    global subscriptions

    # Load the subscription data
    subscriptions = smMAJ.load_subscriptions("/content/Subscription_Info.txt")

    return subscriptions


def valid_customer(customer_id: str):

    global subscriptions

    # error checker
    if len(customer_id) == 4:
        premium = smMAJ.check_subscription(customer_id, subscriptions)
        return premium # bool
    return None



In [99]:
"""
DIYBooking
Last edited on: 2026/11/11
             By: F613550
Cell description:
booking DIY sessions
"""



'\nDIYBooking\nLast edited on: 2026/11/11\n             By: F613550\nCell description:\nbooking DIY sessions\n'

In [100]:
"""
toolRent
Last edited on: 2026/11/11
             By: F613550
Cell description:
    • Prompts the store manager for the customer's ID and the ID of the tool(s)
      they wish to rent.
    • Rental_History.txt: Stores initial example data for the rental history of
      tools. Populate this text file with realistic data (minimum 50 records).
      You may write a program to generate this data.
    •To rent a tool, the store manager should provide a customer ID and the tool's ID number. Your program should then:
    1. Validate the input and use the provided ‘subscriptionManager.pyc’ to check the customer's subscription status. Return an error message if either is not valid.
    2. Depending on the tool's availability and the customer's subscription limit, either allow the manager to rent the tool by updating the related records in the database or return an appropriate message indicating why the rental cannot proceed
"""

def currently_rented():
    return False # will make at a later point


def valid_tool(tool_id):

    global tool_info_list

    if tool_id in tool_info_list.get("ID"): # tool exists?
        if tool_id not in currently_rented(): # tool rented out?
            return True
    return False # tool is in Tool_Info.txt and available


def get_tool_name(tool_id):
    global tool_info_list
    for tool in tool_info_list:
        if tool["ID"] == tool_id:
            return tool["Name"]
    return None


 # for now we will just rent out for 2 weeks from todays date
def update_rental_history(customer_id, tool_id):
    history_path = "/content/Rental_History.txt"

    try:
        with open(history_path, "a") as file:

            tool_name = get_tool_name(tool_id)
            start_date = date.today()
            end_date = start_date + timedelta(weeks=2)
            new_line = f"{tool_name},{start_date},{end_date},{customer_id}\n"
            file.write(new_line)
            return True
    except error as e:
        print(f"An error {e} occured.")
        return False


def toolRent():

    # input boxes
    customer_box = widgets.Text(
        value="",
        placeholder="Enter Customer ID",
        description="Customer ID:",
        layout=widgets.Layout(width="350px")
    )

    tool_box = widgets.Text(
        value="",
        placeholder="Enter Tool ID",
        description="Tool ID:",
        layout=widgets.Layout(width="350px")
    )

    # button
    rent_button = widgets.Button(
        description="Rent Tool",
        button_style="success",
        layout=widgets.Layout(width="200px")
    )

    # when button is clicked
    def on_rent_clicked(b):
        with output:
            clear_output()
            customer_id = customer_box.value.strip()
            tool_id = tool_box.value.strip()

            # Validation
            if not customer_id or not tool_id:
                print("Please enter both Customer ID and Tool ID.")
                return

            if not valid_customer(customer_id):
                print("Invalid customer ID or customer does not have an active subscription.")
                return

            if not valid_tool: #tool available?
                print("Invalid tool ID or tool is already rented out")
                return

            # rent tool to customer
            success = update_rental_history(customer_id, tool_id)

            if success:
                print(f"Success! Tool {tool_id} has been rented to customer {customer_id} for 2 weeks.")
            else:
                print("Failed to update rental history.")

    output = widgets.Output()
    rent_button.on_click(on_rent_clicked)

    ui = widgets.VBox([
        widgets.HTML("Tool Rental"),
        customer_box,
        tool_box,
        rent_button,
        output
    ])

    display(ui)


In [101]:
"""
menu

Last edited on: 2026/11/11
             By: F613550
Cell description:
    This cell contains the main program flow and supporting functions.
    • Installs required packages and imports all necessary libraries.
    • upload_files checks whether the required data files are already present.
    • setup runs the one-time initialisation steps
    • manager_login currently just returns True (placeholder for future login).
    • menu starts the program by calling the login check and then launching
      the tool search interface.
    • Calling setup() and menu() to run the program.
"""

# installation and imports
!pip install -q rapidfuzz ipywidgets

from google.colab import files
import ipywidgets as widgets
from IPython.display import display, clear_output
from rapidfuzz import process, fuzz
from os import error
from pathlib import Path
from datetime import date, timedelta



def upload_files(): # Upload all files needed with error handling

    f_needed = [
        "Bookings.txt",
        "feedbackManager.pyc",
        "Rental_History.txt",
        "Subscription_Info.txt",
        "subscriptionManager.pyc",
        "Tool_Feedback.txt",
        "Tool_Info.txt"
    ]

    while True:
        # Check for missing files
        missing_f = [f for f in f_needed if not Path(f"/content/{f}").exists()]

        if not missing_f:
            print("All required files are already present. Skipping upload.")
            break

        print("Missing files:", missing_f)
        print("Please upload the missing files...")

        try:
            uploaded = files.upload()

            if not uploaded:
                print("Upload was cancelled or no files were selected.")
                break
            else:
                for filename in uploaded.keys():
                    print(f"File {filename} uploaded successfully.")

        except Exception as e:
            print(f"Error {e} occurred during the upload.")
            print("Please check the file name and location and try again.")


def setup():    #is first thing to run and is only run once

    global smMAJ, fmMAJ

    remove_colab_sample_data()
    upload_files()
    tool_info_setup()
    # imports need to be global and files must be uploaded first
    import subscriptionManager as smMAJ
    import feedbackManager as fmMAJ
    subscription_setup()
    #feedback_setup()

def manager_login():
    return True


def menu():

    manager_login()
    toolSearch()
    toolRent()




setup()
menu()

All required files are already present. Skipping upload.
